# A real instrument change: the 2018 and 2019 cohorts of Ho et al.

Ho et al. measured the 2019 cohort after a documented loss of optical efficiency in their
instrument, which they compensated by doubling the integration time. This notebook asks
whether the audit sees that change. The quantities it reports are quoted in Section 3.5 of the
SoftwareX manuscript and in the response to Reviewer 1, comment 4.

**Design.** The model is the paper's model, fitted on 10 isolates of the 2018 cohort.
It is compared on two sets of spectra it has never seen:

- the 5 held-out **qc isolates of 2018**: unseen patients, same instrument state;
- the 25 isolates of **2019**: unseen patients, after the instrument change.

Both sides are new patients and have the same five classes, so the contrast is the
acquisition, not the biology. The comparison is per spectrum and per batch of 100 spectra
(one isolate in 2019; a quarter of an isolate in 2018). It is repeated over five patient
splits (`SPLIT_REPEAT = 0–4`).

**Requirements.** `pypasi >= 0.5.5` and the Ho et al. `.npy` files in `Bacteria/`. Runtime is about 2 minutes.

## 0 · Setup (shared with the other revision notebooks)

In [1]:
# --- shared setup: the data, the patient-disjoint split and the defects of the paper ---
from pathlib import Path
import time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
import pypasi
from pypasi import BandNegotiationClassifier, BandSet, ControlProfile, PLSDA, Triage, degrade
from pypasi.datasets import load_bacteria

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Control limits built from a random split")
assert tuple(int(p) for p in pypasi.__version__.split(".")[:3]) >= (0, 5, 5), "needs pypasi >= 0.5.5"

DATA_DIR = Path("Bacteria")
SEED = 0
# the configuration of softwarex_examples.ipynb; every function below defaults to it
N_BANDS, TOPOLOGY, REGIME, N_COMPONENTS, SCALE_NAME = 7, "complete", "H1", 10, "SNV"
CALIBRATION, CAL_SPLIT = "auto", 0.25
REVIEW_RATE, NOVELTY_Q = 0.10, 0.99
K_SIGMA, LASER_NM, PROFILE_STATS = 3.0, 633.0, ("mean_stress", "band_confidence")
BATCH_N, LOCALISE_REPEATS = 100, 6
SPECTRA_PER_PATIENT = 400


def minmax(X):
    X = np.asarray(X, float)
    lo, hi = X.min(1, keepdims=True), X.max(1, keepdims=True)
    return (X - lo) / np.where(hi > lo, hi - lo, 1.0)


def snv(X):
    X = np.asarray(X, float)
    s = X.std(1, keepdims=True)
    return (X - X.mean(1, keepdims=True)) / np.where(s > 0, s, 1.0)


SCALERS = {"SNV": snv, "min-max": lambda X: np.asarray(X, float)}

data = load_bacteria(DATA_DIR, normalize=False)
AXIS = data.axis
MM_all, y_all = minmax(data.X_dev), data.y_dev      # 2018 cohort, per-spectrum min-max
MM_B, y_B = minmax(data.X_test), data.y_test        # 2019 cohort (external)

# Patient = block of 400 consecutive spectra; verified, never assumed
groups = np.arange(len(y_all)) // SPECTRA_PER_PATIENT
for g in np.unique(groups):
    assert np.unique(y_all[groups == g]).size == 1, "row order is not grouped by patient"
N_PAT = groups.max() + 1
PAT_CLASS = np.array([y_all[groups == g][0] for g in range(N_PAT)])

BAND_RANGE = (400.0, min(1800.0, float(AXIS.max())))
REF_BANDS = BandSet.equal_width(AXIS, 7, lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
REF_EDGES = [(b.lo, b.hi) for b in REF_BANDS.bands]   # where local defects are planted


def _take(pool, per_class, rng):
    taken = []
    for c in np.unique(PAT_CLASS[pool]):
        taken.extend(rng.choice(pool[PAT_CLASS[pool] == c], per_class, replace=False))
    taken = np.array(sorted(taken))
    return taken, np.setdiff1d(pool, taken)


def make_split(split_repeat=0):
    """The paper's split (fit 10 | limits 5 | select 5 | qc 5 isolates) and its defects."""
    rng = np.random.default_rng(SEED + split_repeat)
    pool = np.arange(N_PAT)
    p_qc, pool = _take(pool, 1, rng)
    p_sel, pool = _take(pool, 1, rng)
    p_lim, pool = _take(pool, 1, rng)
    rows = lambda p: np.flatnonzero(np.isin(groups, p))  # noqa: E731
    ix = {"fit": rows(pool), "lim": rows(p_lim), "sel": rows(p_sel), "qc": rows(p_qc)}
    ix["ref"] = np.sort(np.concatenate([ix["lim"], ix["sel"], ix["qc"]]))
    return {
        "repeat": split_repeat, "p_qc": p_qc, "ix": ix,
        "chart_patients": np.sort(np.concatenate([p_lim, p_sel])),
        "sel_def": degrade.plant(MM_all[ix["sel"]], AXIS, edges=REF_EDGES, random_state=SEED + 77),
        "qc_def": degrade.plant(MM_all[ix["qc"]], AXIS, edges=REF_EDGES, random_state=SEED),
    }


def binf(X, f):
    """Average f adjacent channels: a coarser detector sampling."""
    X = np.asarray(X, float)
    if f == 1:
        return X
    n = (X.shape[-1] // f) * f
    return X[..., :n].reshape(*X.shape[:-1], n // f, f).mean(-1)


def fit_model(sp, *, n_bands=N_BANDS, topology=TOPOLOGY, scale=SCALE_NAME, mode="equal",
              regime=REGIME, n_components=N_COMPONENTS, bin_factor=1):
    """Fit the band ensemble and its Triage exactly as the paper does, for one configuration."""
    S = SCALERS[scale]
    prep = lambda X: S(binf(X, bin_factor))  # noqa: E731
    ax = binf(AXIS, bin_factor)
    ix = sp["ix"]
    if mode == "equal":
        bands = BandSet.equal_width(ax, n_bands, lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
    else:
        bands = BandSet.peak_informed(ax, binf(MM_all[ix["fit"]], bin_factor), n_bands,
                                      lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
    lim = (prep(MM_all[ix["lim"]]), y_all[ix["lim"]])
    t0 = time.perf_counter()
    model = BandNegotiationClassifier(
        bands=bands, base_estimator=PLSDA(n_components=n_components), topology=topology,
        regime=regime, calibration=CALIBRATION, calibration_data=lim,
        calibration_split=CAL_SPLIT, random_state=SEED).fit(prep(MM_all[ix["fit"]]), y_all[ix["fit"]])
    t_fit = time.perf_counter() - t0
    tri = Triage(conflict_score="dg", review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,
                 n_components=n_components).fit(
        model, *lim, orientation_data=(prep(sp["sel_def"].X), sp["sel_def"].degraded))
    return {"model": model, "triage": tri, "bands": bands, "prep": prep, "t_fit": t_fit}


def localise(sp, fm, planted_intervals=None, trials=None):
    """Plant a local defect in a batch of 100 spectra of one qc isolate; does the chart name it?

    A hit is a named band that overlaps the planted interval; chance is the share of bands
    that overlap it. By default the seven published intervals, six trials each (42 trials).
    """
    prep, bands, model = fm["prep"], fm["bands"], fm["model"]
    edges, labs = [(b.lo, b.hi) for b in bands.bands], bands.labels
    rngb = np.random.default_rng(SEED + 11)
    ref = [prep(MM_all[np.sort(rngb.choice(np.flatnonzero(groups == g), BATCH_N, replace=False))])
           for g in sp["chart_patients"]]
    prof = ControlProfile.fit(model, batches=ref, k=K_SIGMA, laser_nm=LASER_NM,
                              statistics=PROFILE_STATS, random_state=SEED)
    if trials is None:
        trials = [(lo, hi, np.random.default_rng(500 * t + rep))
                  for t, (lo, hi) in enumerate(REF_EDGES) for rep in range(LOCALISE_REPEATS)]
    hits, chance = [], []
    for lo, hi, r in trials:
        sel = (AXIS >= lo) & (AXIS < hi)
        overlap = [j for j, (a, b) in enumerate(edges) if a < hi and b > lo]
        g = r.choice(sp["p_qc"])
        Xl = MM_all[np.sort(r.choice(np.flatnonzero(groups == g), BATCH_N, replace=False))].copy()
        Xl[:, sel] = Xl[:, sel] * 0.55 + r.normal(0, 0.02, (len(Xl), int(sel.sum())))
        tab = prof.check(prep(Xl), name="trial").table
        named = tab.assign(az=tab["z"].abs()).groupby("label")["az"].max().idxmax()
        hits.append(labs.index(named) in overlap)
        chance.append(len(overlap) / len(edges))
    return float(np.mean(hits)), float(np.mean(chance))


def evaluate(sp, *, loc=True, **config):
    """Every number the guide reports for one configuration on one split."""
    cfg = {"n_bands": N_BANDS, "topology": TOPOLOGY, "scale": SCALE_NAME, "mode": "equal",
           "regime": REGIME, "n_components": N_COMPONENTS, "bin_factor": 1, **config}
    fm = fit_model(sp, **cfg)
    ix, qd, prep = sp["ix"], sp["qc_def"], fm["prep"]
    y_qc = y_all[ix["qc"]]
    t0 = time.perf_counter()
    T = fm["triage"].assess(prep(qd.X), y_qc).table
    t_audit = time.perf_counter() - t0
    c = T["conflict"].to_numpy()
    fam = {k: roc_auc_score(qd.degraded[(qd.kind == k) | (qd.degraded == 0)],
                            c[(qd.kind == k) | (qd.degraded == 0)]) for k in degrade.PAPER_FAMILIES}
    row = {"repeat": sp["repeat"], **cfg,
           "channels_per_band": int(np.median([b.n_features for b in fm["bands"].bands])),
           "conflict_vs_degraded": roc_auc_score(qd.degraded, c),
           "conflict_vs_wrong": roc_auc_score(1 - T["correct"], c),
           "local": fam["local contamination"],
           "global_mean": float(np.mean([fam[k] for k in degrade.PAPER_FAMILIES if k != "local contamination"])),
           "acc_in_cohort": accuracy_score(y_all[ix["ref"]], fm["model"].predict(prep(MM_all[ix["ref"]]))),
           "acc_external": accuracy_score(y_B, fm["model"].predict(prep(MM_B))),
           "t_fit_s": fm["t_fit"], "t_audit_s": t_audit,
           "orientation_margin": fm["triage"].orientation_margin_}
    if loc:
        row["loc_hit"], row["loc_chance"] = localise(sp, fm)
    return row


In [2]:
def check(rows):
    """Compare this run with the values quoted in the manuscript and the response letter."""
    t = pd.DataFrame(rows, columns=["quantity", "this_run", "quoted", "tol"])
    t["ok"] = (t["this_run"] - t["quoted"]).abs() <= t["tol"]
    print(t.round(4).to_string(index=False))
    print("\nALL QUOTED VALUES REPRODUCE" if t["ok"].all() else
          f"\n{(~t['ok']).sum()} VALUES DO NOT REPRODUCE: correct the text or the run")
    return t

## 1 · Five splits

In [3]:
rows = []
for rep in range(5):
    sp = make_split(rep)
    fm = fit_model(sp)
    tri, prep = fm["triage"], fm["prep"]
    y_qc = y_all[sp["ix"]["qc"]]
    T18 = tri.assess(prep(MM_all[sp["ix"]["qc"]]), y_qc).table      # held-out 2018
    T19 = tri.assess(prep(MM_B), y_B).table                          # 2019, after the change
    lab = np.r_[np.zeros(len(T18)), np.ones(len(T19))]
    r = {"split": rep, "acc_2018_heldout": T18["correct"].mean(), "acc_2019": T19["correct"].mean()}
    for name, col, sign in (("conflict", "conflict", 1), ("1 - confidence", "confidence", -1),
                            ("novelty", "novelty", 1)):
        r[name] = roc_auc_score(lab, sign * np.r_[T18[col].to_numpy(), T19[col].to_numpy()])
    # batches of 100 consecutive spectra: one isolate in 2019, a quarter of an isolate in 2018
    b18, b19 = np.arange(len(T18)) // 100, np.arange(len(T19)) // 100
    blab = np.r_[np.zeros(b18.max() + 1), np.ones(b19.max() + 1)]
    for name, col, sign in (("conflict, batch of 100", "conflict", 1),
                            ("1 - confidence, batch of 100", "confidence", -1)):
        m = np.r_[T18.groupby(b18)[col].mean().to_numpy(), T19.groupby(b19)[col].mean().to_numpy()]
        r[name] = roc_auc_score(blab, sign * m)
    rows.append(r)
change = pd.DataFrame(rows)
print(change.round(3).to_string(index=False))
print("\nmean ± sd over five splits")
print((change.drop(columns="split").mean().round(3).astype(str) + " ± "
       + change.drop(columns="split").std().round(3).astype(str)).to_string())
print(f"\nconflict above chance in {(change['conflict'] > 0.5).sum()} of 5 splits; "
      f"lowest {change['conflict'].min():.3f}")

 split  acc_2018_heldout  acc_2019  conflict  1 - confidence  novelty  conflict, batch of 100  1 - confidence, batch of 100
     0             0.910     0.797     0.743           0.511    0.352                   0.796                         0.536
     1             0.930     0.746     0.710           0.595    0.407                   0.788                         0.634
     2             0.787     0.800     0.815           0.460    0.392                   0.976                         0.462
     3             0.736     0.763     0.689           0.422    0.375                   0.794                         0.380
     4             0.920     0.747     0.789           0.516    0.533                   0.960                         0.516

mean ± sd over five splits
acc_2018_heldout                0.856 ± 0.089
acc_2019                        0.771 ± 0.026
conflict                        0.749 ± 0.053
1 - confidence                  0.501 ± 0.065
novelty                         0.412 ± 0.07

## 2 · Every value quoted, checked

In [4]:
M = change.mean(numeric_only=True)
claims = check([
    ("conflict, per spectrum (mean)",          M["conflict"], 0.749, 0.005),
    ("conflict, per spectrum (sd)",            change["conflict"].std(), 0.053, 0.005),
    ("conflict, lowest split",                 change["conflict"].min(), 0.689, 0.005),
    ("1 - confidence, per spectrum (mean)",    M["1 - confidence"], 0.501, 0.005),
    ("novelty, per spectrum (mean)",           M["novelty"], 0.412, 0.005),
    ("conflict, batch of 100 (mean)",          M["conflict, batch of 100"], 0.863, 0.005),
    ("1 - confidence, batch of 100 (mean)",    M["1 - confidence, batch of 100"], 0.506, 0.005),
])
change.to_csv("instrument_change.csv", index=False)

                           quantity  this_run  quoted   tol   ok
      conflict, per spectrum (mean)    0.7489   0.749 0.005 True
        conflict, per spectrum (sd)    0.0528   0.053 0.005 True
             conflict, lowest split    0.6886   0.689 0.005 True
1 - confidence, per spectrum (mean)    0.5008   0.501 0.005 True
       novelty, per spectrum (mean)    0.4118   0.412 0.005 True
      conflict, batch of 100 (mean)    0.8628   0.863 0.005 True
1 - confidence, batch of 100 (mean)    0.5056   0.506 0.005 True

ALL QUOTED VALUES REPRODUCE
